In [5]:
import yaml

import torch
import torch.nn as nn
import torch.quantization as tq

In [1]:
from onnxruntime.quantization import quantize_static, QuantFormat, QuantType, CalibrationDataReader
import onnx
import numpy as np

In [6]:
from mlp_for_quantization import MLP

In [7]:
class QuantModel(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.quant = torch.quantization.QuantStub()
        self.model = model
        self.dequant = torch.quantization.DeQuantStub()
    
    def forward(self, x):
        x = self.quant(x)
        x = self.model(x)
        x = self.dequant(x)
        return x

In [8]:
with open('checkpoints/config_narrow.yaml', 'r', encoding='UTF-8') as handle:
    config = yaml.safe_load(handle)

model = MLP(**config['model'])

model_fp32 = QuantModel(model).eval()

# Use fbgemm for x86 CPUs
model_fp32.qconfig = tq.get_default_qconfig("fbgemm")

print("=== QCONFIG ===")
print(model_fp32.qconfig)

example_inputs = list(torch.randn(1, 50, 6) for _ in range(100))
tq.prepare(model_fp32, inplace=True)

with torch.no_grad():
    for data in example_inputs:
        model_fp32(data)

model_int8 = tq.convert(model_fp32, inplace=False)
print(model_int8)

with torch.no_grad():
    for i, data in enumerate(example_inputs):
        output = model_int8(data)

=== QCONFIG ===
QConfig(activation=functools.partial(<class 'torch.ao.quantization.observer.HistogramObserver'>, reduce_range=True){}, weight=functools.partial(<class 'torch.ao.quantization.observer.PerChannelMinMaxObserver'>, dtype=torch.qint8, qscheme=torch.per_channel_symmetric){})
QuantModel(
  (quant): Quantize(scale=tensor([0.0628]), zero_point=tensor([66]), dtype=torch.quint8)
  (model): MLP(
    (embed): Sequential(
      (0): Identity()
      (1): QuantizedLinear(in_features=6, out_features=128, scale=0.04395966976881027, zero_point=66, qscheme=torch.per_channel_affine)
      (2): QuantizedLeakyReLU(negative_slope=0.1)
      (3): Identity()
      (4): QuantizedLinear(in_features=128, out_features=128, scale=0.020298566669225693, zero_point=62, qscheme=torch.per_channel_affine)
      (5): QuantizedLeakyReLU(negative_slope=0.1)
    )
    (solvers): ModuleList(
      (0): SubsetSolver(
        (model): Sequential(
          (0): Identity()
          (1): QuantizedLinear(in_featur